In [1]:
suppressPackageStartupMessages({
  suppressWarnings({
    library(Seurat)
    library(MAST)
    library(dplyr)
    library(data.table)
    library(ggplot2)
    library(schard)
    library(argparse)
  })
})


In [2]:
input<-'/data1st2/junyi/output/atac1112/subset/region_nt/PFC_PFC_2k.h5ad'
output<-'/data2st2/junyi/output/test/dar'

In [3]:
seo = schard::h5ad2seurat(input)
#seo <- subset(seo, subset = !(sample %in% c("MW26A_PFC", "MC25A_PFC")))
df_qc <- read.csv('/data2st1/junyi/output/atac0627/frac_qc.csv', row.names = 1)
df_qc$sample <- rownames(df_qc)
metadata <- seo@meta.data
cell_groups <- unique(metadata[["celltype.L2"]])
metadata$cell_bc <- rownames(metadata)
meta2 <- merge(metadata, df_qc, by = "sample", all.x = TRUE)
rownames(meta2) <- meta2$cell_bc
seo@meta.data <- meta2
colnames(seo@meta.data)[colnames(seo@meta.data) == "Fraction.of.high.quality.fragments.overlapping.peaks"] <- "frac_peak"
setwd(output)

In [4]:


perform_mast_celltype_specific <- function(
    seurat_obj,
    group.by = "celltype.L2",    # cell type column
    batch.by = NULL,
    freq_expressed = 0.1,
    save.as.tmp = TRUE
){
  library(MAST)
  library(data.table)

  metadata <- seurat_obj@meta.data
  celltypes <- unique(metadata[[group.by]])

  all_results <- data.frame()

  for (ct in celltypes) {

    cat("====================================\n")
    cat("Celltype-specific DAR for:", ct, "\n")

    # Define labels
    metadata$compare_group <- ifelse(metadata[[group.by]] == ct, ct, "others")
    metadata$compare_group <- factor(metadata$compare_group, levels=c(ct,"others"))
    seurat_obj$compare_group <- metadata$compare_group

    # Subset only relevant cells (to reduce memory)
    # keep_cells <- rownames(metadata)
    # seo_subset <- subset(seurat_obj, cells = keep_cells)

    # Filter peaks expressed in >=10 cells
    exprs <- GetAssayData(seurat_obj, slot = "counts")
    keep <- rowSums(exprs > 0) >= 10
    seo_subset <- seurat_obj[keep, ]

    expr_matrix <- GetAssayData(seo_subset, slot = "data")
    anno <- seo_subset@meta.data

    # Build MAST object
    sca <- FromMatrix(as.matrix(expr_matrix), cData = anno)

    # Frequency filter
    select.genes <- freq(sca) > freq_expressed
    sca <- sca[select.genes, ]

    # Add CDR2 (ngenes)
    cdr2 <- colSums(assay(sca) > 0)
    colData(sca)$ngeneson <- scale(cdr2)

    # Add batch effect if exists
    # if (!is.null(batch.by)) {
    #   colData(sca)$batch <- factor(colData(sca)[[batch.by]])
    #   zlm_model <- zlm(~ compare_group + ngeneson + batch, sca)
    # } else {
    zlm_model <- zlm(~ compare_group + ngeneson, sca)
    # }

    # MAST contrast test
    contrast <- paste0("compare_groupothers")  
    summary_result <- summary(zlm_model, doLRT = contrast)
    dt <- summary_result$datatable

    fcHurdle <- merge(
      dt[contrast == contrast & component == "H", .(primerid, `Pr(>Chisq)`)],
      dt[contrast == contrast & component == "logFC",
         .(primerid, coef, ci.hi, ci.lo)],
      by="primerid"
    )

    if (nrow(fcHurdle)==0) {
      cat("No results for:", ct, "\n")
      next
    }

    fcHurdle[, padjust := p.adjust(`Pr(>Chisq)`, "bonferroni")]
    fcHurdle$celltype <- ct

    all_results <- rbind(all_results, fcHurdle)

    if (save.as.tmp) {
      write.csv(fcHurdle,
                paste0("DAR_celltype_specific_", make.names(ct), ".csv"))
    }
  }

  return(all_results)
}


In [5]:
seo <- subset(seo, subset = celltype.L1_ct != "OPC")
r.dar_celltype <- perform_mast_celltype_specific(
    seo,
    group.by = "celltype.L2",
    batch.by = "date"
)

Celltype-specific DAR for: HPF_CA1_Glut 


Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 239.7 GiB”
`cData` has no wellKey.  I'll make something up.

Assuming data assay in position 1, with name et is log-transformed.


Done!

Combining coefficients and standard errors

Calculating log-fold changes

Calculating likelihood ratio tests

Refitting on reduced model...


Done!



Celltype-specific DAR for: HPF_DG_GC_Glut 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 239.7 GiB”
`cData` has no wellKey.  I'll make something up.

Assuming data assay in position 1, with name et is log-transformed.



: 